In [ ]:
# Cell 1 — Installs
!pip install ultralytics kornia --quiet

In [ ]:
# Cell 2 — Imports
import cv2
import random
import numpy as np
import torch
import torch.nn.functional as F
import kornia
import kornia.augmentation as K
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

In [ ]:
# Cell 3 — Config
# ── Training ──────────────────────────────────────────
PATCH_H       = 200      # patch height in pixels
PATCH_W       = 150      # patch width in pixels
PATCH_SCALE   = 0.3      # fraction of detected person width used for patch
SHIRT_TOP     = 0.30     # top of shirt region as fraction of person bbox height
SHIRT_BOT     = 0.70     # bottom of shirt region as fraction of person bbox height
EPSILON       = 0.15     # max pixel delta from initialization (0–1 range)
LR            = 0.01     # Adam learning rate
NUM_STEPS     = 200      # total training steps (use 30 for debug run)
EOT_N         = 8        # EoT transforms per step
TOP_K         = 50       # top anchors to target in confidence loss
CONF_LOSS_W   = 0.6      # weight for Top-K confidence loss
SEG_LOSS_W    = 0.4      # weight for mask suppression loss

# ── Input ──────────────────────────────────────────────
# Upload one image to Colab and set this path.
# Any image with a clearly visible person works (e.g., someone walking).
IMAGE_PATH    = "person.jpg"

# ── Model ──────────────────────────────────────────────
YOLO_MODEL    = "yolov8n-seg"   # nano for speed; swap yolov8x-seg for eval
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 4 — Load YOLOv8-seg, freeze weights
yolo = YOLO(YOLO_MODEL)
torch_model = yolo.model.to(DEVICE)
torch_model.eval()

# Freeze all weights — only patch pixels will be updated
for p in torch_model.parameters():
    p.requires_grad_(False)

print(f"Model loaded: {YOLO_MODEL} | Parameters frozen: {sum(p.numel() for p in torch_model.parameters()):,}")

In [ ]:
# Cell 5 — Raw forward pass helper
def yolo_raw_forward(img_tensor):
    """
    img_tensor: [1, 3, 640, 640] float32 on DEVICE, values in [0, 1]
    Returns dict:
        'preds': [1, 116, 8400]  — 4 box + 80 class logits + 32 mask coeffs per anchor
        'proto': [1, 32, 160, 160] — mask prototype basis
    """
    out = torch_model(img_tensor)
    return {'preds': out[0], 'proto': out[1][0]}


def img_to_tensor(img_np):
    """uint8 HxWx3 numpy → [1, 3, 640, 640] float32 tensor on DEVICE"""
    img = cv2.resize(img_np, (640, 640))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    return t.unsqueeze(0).to(DEVICE)

In [ ]:
# Cell 6 — Shirt detection and patch placement
def detect_shirt_region(img_np):
    """
    Detects the highest-confidence person and returns:
        shirt_box_img: (x1, y1, x2, y2) in 640x640 image coordinates
        shirt_box_mask: (x1, y1, x2, y2) in 160x160 mask coordinates
    Returns (None, None) if no person detected.
    """
    with torch.no_grad():
        results = yolo(cv2.resize(img_np, (640, 640)), verbose=False)

    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return None, None

    # Find highest-confidence person detection (class 0)
    boxes = results[0].boxes
    person_mask = (boxes.cls == 0)
    if not person_mask.any():
        return None, None

    best_idx = boxes.conf[person_mask].argmax()
    box = boxes.xyxy[person_mask][best_idx].cpu().numpy()  # [x1, y1, x2, y2]

    x1, y1, x2, y2 = box
    h = y2 - y1
    # Shirt region: middle vertical band of the person bbox
    sx1 = int(x1)
    sy1 = int(y1 + h * SHIRT_TOP)
    sx2 = int(x2)
    sy2 = int(y1 + h * SHIRT_BOT)

    # Scale to mask resolution (160x160 = 640/4)
    scale = 160 / 640
    mx1, my1 = int(sx1 * scale), int(sy1 * scale)
    mx2, my2 = int(sx2 * scale), int(sy2 * scale)

    # Clamp to valid range
    mx1, my1 = max(0, mx1), max(0, my1)
    mx2, my2 = min(159, mx2), min(159, my2)

    if mx2 <= mx1 or my2 <= my1:
        return None, None

    return (sx1, sy1, sx2, sy2), (mx1, my1, mx2, my2)


def paste_patch(img_tensor, patch, shirt_box):
    """
    img_tensor: [1, 3, 640, 640] float32
    patch: [1, 3, PATCH_H, PATCH_W] float32, differentiable
    shirt_box: (x1, y1, x2, y2) in image coords
    Returns: patched image tensor [1, 3, 640, 640]
    """
    x1, y1, x2, y2 = int(shirt_box[0]), int(shirt_box[1]), int(shirt_box[2]), int(shirt_box[3])
    w, h = x2 - x1, y2 - y1
    if w <= 0 or h <= 0:
        return img_tensor

    # Resize patch to shirt region dimensions
    patch_resized = F.interpolate(patch, size=(h, w), mode='bilinear', align_corners=False)

    # Clone and paste (preserve gradient graph through patch_resized)
    patched = img_tensor.clone()
    patched[:, :, y1:y2, x1:x2] = patch_resized
    return patched

In [ ]:
# Cell 7 — Loss functions
def topk_conf_loss(raw, k=TOP_K):
    """
    Minimize the mean confidence of the k highest-confidence anchors.
    Targets wherever YOLO actually fires (head/shoulders ~0.85 conf),
    not the shirt region which has near-zero confidence and dead gradients.
    """
    preds = raw['preds']                         # [1, 116, 8400]
    class_logits = preds[0, 4:84, :]             # [80, 8400]
    max_logits = class_logits.max(dim=0)[0]      # [8400]
    conf = torch.sigmoid(max_logits)             # [8400]
    top_k_vals = torch.topk(conf, k=k).values    # [k]
    return top_k_vals.mean()

In [ ]:
# Cell 8 — Mask suppression loss and combined loss
def mask_suppression_loss(raw, patch_box_mask):
    """
    Assemble the segmentation mask for the highest-confidence detection,
    then minimize its probability in the patch region.
    Directly degrades silhouette quality where the patch sits.
    """
    preds = raw['preds']                              # [1, 116, 8400]
    proto = raw['proto']                              # [1, 32, 160, 160]

    class_logits = preds[0, 4:84, :]
    conf = torch.sigmoid(class_logits.max(0)[0])      # [8400]
    best = conf.argmax()

    # Skip if no confident detection — return zero loss with grad
    if conf[best].item() < 0.1:
        return preds[0, 84:, 0].sum() * 0.0  # zero in-graph (not a leaf tensor) so patch.grad stays valid

    mask_coeff = preds[0, 84:, best]                  # [32]
    proto_flat = proto[0].reshape(32, -1)             # [32, 160*160]
    mask_logits = (mask_coeff @ proto_flat).reshape(160, 160)   # [160, 160]
    mask_prob = torch.sigmoid(mask_logits)            # [160, 160]

    x1, y1, x2, y2 = patch_box_mask
    patch_region = mask_prob[y1:y2, x1:x2]

    if patch_region.numel() == 0:
        return preds[0, 84:, 0].sum() * 0.0  # zero in-graph (not a leaf tensor) so patch.grad stays valid

    return patch_region.mean()


def combined_loss(raw, patch_box_mask):
    """Weighted sum of confidence suppression and mask suppression losses."""
    conf_loss = topk_conf_loss(raw)
    seg_loss  = mask_suppression_loss(raw, patch_box_mask)
    return CONF_LOSS_W * conf_loss + SEG_LOSS_W * seg_loss

In [ ]:
# Cell 9 — Differentiable EoT transforms via kornia
def apply_eot_transforms(patch):
    """
    patch: [1, 3, PATCH_H, PATCH_W] float32, grad-enabled
    Returns: transformed patch, same shape, gradient preserved
    All transforms use kornia or raw torch — no PIL/numpy (which break autograd).
    """
    p = patch  # keep reference for conditional applications

    # Brightness: multiply by scalar in [0.6, 1.4]
    if random.random() < 0.5:
        factor = random.uniform(0.6, 1.4)
        p = (p * factor).clamp(0, 1)

    # Hue/Saturation jitter ±10%
    if random.random() < 0.5:
        p = K.ColorJitter(hue=0.1, saturation=0.1, p=1.0)(p)

    # Rotation ±15°
    if random.random() < 0.5:
        angle = torch.tensor([random.uniform(-15, 15)], device=DEVICE)
        p = kornia.geometry.rotate(p, angle)

    # Perspective warp (slight fabric curvature)
    if random.random() < 0.5:
        p = K.RandomPerspective(distortion_scale=0.2, p=1.0)(p)

    # Motion blur
    if random.random() < 0.5:
        ksize = random.choice([3, 5, 7])
        blur_angle = random.uniform(0, 360)
        p = kornia.filters.motion_blur(p, kernel_size=ksize, angle=blur_angle, direction=0.0)

    # Gaussian noise (pure torch, stays differentiable)
    if random.random() < 0.5:
        sigma = random.uniform(0, 10) / 255.0
        p = p + torch.randn_like(p) * sigma

    return p.clamp(0, 1)

In [ ]:
# Cell 10 — Baseline: measure confidence before training
img = cv2.imread(IMAGE_PATH)
if img is None:
    raise FileNotFoundError(f"Could not load {IMAGE_PATH} — check Colab upload")

img_tensor = img_to_tensor(img)

with torch.no_grad():
    raw_baseline = yolo_raw_forward(img_tensor)

baseline_conf = topk_conf_loss(raw_baseline).item()
print(f"Baseline Top-{TOP_K} mean confidence: {baseline_conf:.4f}")
# Expected: ~0.6–0.9 for a clear person image

In [ ]:
# Cell 11 — Initialize patch
torch.manual_seed(42)
patch = torch.empty(1, 3, PATCH_H, PATCH_W, device=DEVICE).uniform_(0.4, 0.6)
patch.requires_grad_(True)
patch_init = patch.data.clone()  # store for epsilon projection

optimizer = torch.optim.Adam([patch], lr=LR)
print(f"Patch initialized: shape {list(patch.shape)}, range [{patch.min():.2f}, {patch.max():.2f}]")

In [ ]:
# Cell 12 — Training loop
img = cv2.imread(IMAGE_PATH)
box_img, box_mask = detect_shirt_region(img)
if box_img is None:
    raise RuntimeError("No person detected in training image — use a clearer image")

img_tensor = img_to_tensor(img)  # [1, 3, 640, 640], no grad needed
loss_history = []

for step in range(NUM_STEPS):
    optimizer.zero_grad()
    step_loss = 0.0

    for _ in range(EOT_N):
        aug_patch = apply_eot_transforms(patch)
        patched_img = paste_patch(img_tensor, aug_patch, box_img)
        raw = yolo_raw_forward(patched_img)
        loss = combined_loss(raw, box_mask) / EOT_N
        loss.backward()
        step_loss += loss.item()

    optimizer.step()

    # Project patch to epsilon ball around initialization
    with torch.no_grad():
        delta = (patch.data - patch_init).clamp(-EPSILON, EPSILON)
        patch.data = (patch_init + delta).clamp(0, 1)

    mean_loss = step_loss / EOT_N
    loss_history.append(mean_loss)

    if step % 10 == 0:
        print(f"Step {step:3d}/{NUM_STEPS} | loss {mean_loss:.4f}")

print("Training complete.")

In [ ]:
# Cell 13 — Evaluation
img = cv2.imread(IMAGE_PATH)
img_tensor = img_to_tensor(img)
box_img, box_mask = detect_shirt_region(img)

with torch.no_grad():
    # Clean image metrics
    raw_clean = yolo_raw_forward(img_tensor)
    clean_conf = topk_conf_loss(raw_clean).item()
    clean_mask_prob = mask_suppression_loss(raw_clean, box_mask).item()

    # Patched image metrics
    patch_applied = paste_patch(img_tensor, patch, box_img)
    raw_patched = yolo_raw_forward(patch_applied)
    patched_conf = topk_conf_loss(raw_patched).item()
    patched_mask_prob = mask_suppression_loss(raw_patched, box_mask).item()

print("─" * 45)
print(f"{'Metric':<30} {'Clean':>6}  {'Patched':>7}")
print("─" * 45)
print(f"{'Top-K conf (lower=better)':<30} {clean_conf:>6.4f}  {patched_conf:>7.4f}")
print(f"{'Mask prob in patch region':<30} {clean_mask_prob:>6.4f}  {patched_mask_prob:>7.4f}")
conf_drop_pct = (clean_conf - patched_conf) / clean_conf * 100
print("─" * 45)
print(f"Confidence drop: {conf_drop_pct:.1f}%")

In [ ]:
# Cell 14 — Plots
img = cv2.imread(IMAGE_PATH)
box_img, _ = detect_shirt_region(img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loss curve
axes[0].plot(loss_history)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True)

# Clean image with YOLO overlay
results_clean = yolo(cv2.resize(cv2.imread(IMAGE_PATH), (640, 640)), verbose=False)
axes[1].imshow(cv2.cvtColor(results_clean[0].plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title("Clean — YOLO detections")
axes[1].axis('off')

# Patched image with YOLO overlay
patch_np = patch.detach().cpu().squeeze().permute(1, 2, 0).numpy()
patch_np = (patch_np * 255).astype(np.uint8)
img_patched_np = cv2.resize(cv2.imread(IMAGE_PATH), (640, 640)).copy()
x1, y1, x2, y2 = box_img
patch_resized = cv2.resize(patch_np, (x2 - x1, y2 - y1))
img_patched_np[y1:y2, x1:x2] = cv2.cvtColor(patch_resized, cv2.COLOR_RGB2BGR)
results_patched = yolo(img_patched_np, verbose=False)
axes[2].imshow(cv2.cvtColor(results_patched[0].plot(), cv2.COLOR_BGR2RGB))
axes[2].set_title("Patched — YOLO detections")
axes[2].axis('off')

plt.tight_layout()
plt.savefig("eval_v2.png", dpi=150)
plt.show()
print("Saved eval_v2.png")

In [ ]:
# Cell 15 — Save outputs
patch_np = patch.detach().cpu().squeeze().permute(1, 2, 0).numpy()
patch_np = (patch_np * 255).astype(np.uint8)

# Native resolution
cv2.imwrite("patch_v2.png", cv2.cvtColor(patch_np, cv2.COLOR_RGB2BGR))

# Print-ready upscaled
patch_print = cv2.resize(patch_np, (300, 300), interpolation=cv2.INTER_NEAREST)
cv2.imwrite("patch_v2_print.png", cv2.cvtColor(patch_print, cv2.COLOR_RGB2BGR))

print("Saved: patch_v2.png, patch_v2_print.png")
print(f"Patch pixel range: [{patch_np.min()}, {patch_np.max()}]")